# Only SFT Training Notebook (Self-Contained)

Этот ноутбук самодостаточный: содержит весь код для SFT-обучения GPT2, weighted loss, чекпоинт лучшей модели и метрики `Pass@1`, `Pass@k`, `avg_reward_score`.

Запуск на Kaggle: загрузите только CSV-данные и настройте пути в секции `Run Config`.


## 1. Imports and Global Settings

In [ ]:
import ast
import json
import math
import os
import re
import signal
import subprocess
import sys
import tempfile
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizerBase,
    get_cosine_schedule_with_warmup,
)

try:
    import resource  # POSIX-only
except Exception:
    resource = None


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## 2. Prompt Tokens and Formatting

In [ ]:
QUESTION_TOKEN = "<|question|>"
EXAMPLE_TOKEN = "<|example|>"
REASONING_TOKEN = "<|reasoning|>"
CODE_TOKEN = "<|code|>"

SPECIAL_TOKENS = [
    QUESTION_TOKEN,
    EXAMPLE_TOKEN,
    REASONING_TOKEN,
    CODE_TOKEN,
]


class PromptFormatter:
    @staticmethod
    def format_input_prompt(question: str, example: str) -> str:
        return (
            f"{QUESTION_TOKEN}\n"
            f"{question.strip()}\n"
            f"{EXAMPLE_TOKEN}\n"
            f"{example.strip()}\n"
            f"{REASONING_TOKEN}\n"
        )

    @staticmethod
    def format_output_target(reasoning: str, solution: str) -> str:
        return (
            f"{reasoning.strip()}\n\n"
            f"{CODE_TOKEN}\n"
            f"{solution.strip()}"
        )

    @staticmethod
    def format_sft_sample(question: str, example: str, reasoning: str, solution: str) -> str:
        return (
            PromptFormatter.format_input_prompt(question=question, example=example)
            + PromptFormatter.format_output_target(reasoning=reasoning, solution=solution)
        )

    @staticmethod
    def format_generation_prompt(question: str, example: str) -> str:
        return PromptFormatter.format_input_prompt(question=question, example=example)


def parse_domain_csvs(raw_value: str) -> Dict[str, str]:
    items = [x.strip() for x in raw_value.split(",") if x.strip()]
    result: Dict[str, str] = {}
    for item in items:
        if ":" not in item:
            raise ValueError(f"Invalid domain spec '{item}'. Use domain:path.csv format.")
        domain, path = item.split(":", maxsplit=1)
        result[domain.strip()] = path.strip()
    if not result:
        raise ValueError("No domain CSVs parsed from domain_csvs.")
    return result


## 3. Model Factory

In [ ]:
@dataclass
class ModelConfig:
    model_name: str = "gpt2"


class ModelFactory:
    def __init__(self, config: ModelConfig) -> None:
        self.config = config

    def load_model(self, model_name_or_path: str):
        return AutoModelForCausalLM.from_pretrained(model_name_or_path)

    def build(self):
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        model = self.load_model(self.config.model_name)

        if tokenizer.pad_token is None:
            if tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token
            else:
                tokenizer.add_special_tokens({"pad_token": "[PAD]"})

        tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
        model.config.pad_token_id = tokenizer.pad_token_id
        model.resize_token_embeddings(len(tokenizer))

        return model, tokenizer


## 4. Dataset and Dataloaders

In [ ]:
@dataclass
class DataConfig:
    max_length: int = 768
    train_batch_size: int = 2
    eval_batch_size: int = 2
    num_workers: int = 0
    reasoning_weight: float = 1.0
    code_weight: float = 1.0
    spec_weight: float = 1.0


class CausalLMDataset(Dataset):
    def __init__(
        self,
        rows: List[dict],
        tokenizer,
        max_length: int,
        reasoning_weight: float = 1.0,
        code_weight: float = 1.0,
        spec_weight: float = 1.0,
    ) -> None:
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.reasoning_weight = float(reasoning_weight)
        self.code_weight = float(code_weight)
        self.spec_weight = float(spec_weight)
        self.eos_token_id = tokenizer.eos_token_id

    def __len__(self) -> int:
        return len(self.rows)

    def _encode_target_with_weights(self, row: dict) -> Tuple[List[int], List[float]]:
        reasoning_text = f"{row['reasoning'].strip()}\n\n"
        code_marker_text = f"{CODE_TOKEN}\n"
        code_text = row["solution"].strip()

        reasoning_ids = self.tokenizer(reasoning_text, add_special_tokens=False)["input_ids"]
        code_marker_ids = self.tokenizer(code_marker_text, add_special_tokens=False)["input_ids"]
        code_ids = self.tokenizer(code_text, add_special_tokens=False)["input_ids"]

        target_ids = reasoning_ids + code_marker_ids + code_ids
        target_weights = (
            [self.reasoning_weight] * len(reasoning_ids)
            + [self.spec_weight] * len(code_marker_ids)
            + [self.code_weight] * len(code_ids)
        )

        if self.eos_token_id is not None:
            target_ids.append(self.eos_token_id)
            target_weights.append(self.code_weight)

        return target_ids, target_weights

    def __getitem__(self, idx: int):
        row = self.rows[idx]

        prompt_text = PromptFormatter.format_input_prompt(
            question=row["question"],
            example=row.get("example", ""),
        )
        prompt_ids = self.tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        target_ids, target_weights = self._encode_target_with_weights(row)

        input_ids = prompt_ids + target_ids
        labels = input_ids.copy()
        token_weights = [0.0] * len(prompt_ids) + target_weights

        input_ids = input_ids[: self.max_length]
        labels = labels[: self.max_length]
        token_weights = token_weights[: self.max_length]

        for i in range(min(len(prompt_ids), len(labels))):
            labels[i] = -100

        input_ids = torch.tensor(input_ids, dtype=torch.long)
        attention_mask = torch.ones_like(input_ids)
        labels = torch.tensor(labels, dtype=torch.long)
        token_weights = torch.tensor(token_weights, dtype=torch.float32)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "token_weights": token_weights,
        }


class CausalLMDataCollator:
    def __init__(self, tokenizer) -> None:
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids = [item["input_ids"] for item in batch]
        attention_masks = [item["attention_mask"] for item in batch]
        labels = [item["labels"] for item in batch]
        token_weights = [item["token_weights"] for item in batch]

        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0,
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )
        token_weights = torch.nn.utils.rnn.pad_sequence(
            token_weights,
            batch_first=True,
            padding_value=0.0,
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "token_weights": token_weights,
        }


class ReasoningDataModule:
    def __init__(self, config: DataConfig, tokenizer) -> None:
        self.config = config
        self.tokenizer = tokenizer
        self.collator = CausalLMDataCollator(tokenizer=tokenizer)

    @staticmethod
    def _normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
        rename_map = {"reasonings": "reasoning", "solutions": "solution"}
        df = df.rename(columns=rename_map)

        if "example" not in df.columns:
            df["example"] = ""
        if "reasoning" not in df.columns:
            df["reasoning"] = ""

        required_columns = ["question", "example", "reasoning", "solution"]
        missing = [col for col in required_columns if col not in df.columns]
        if missing:
            raise ValueError(
                f"CSV does not contain required columns: {missing}. Found columns: {list(df.columns)}"
            )

        return df[required_columns].fillna("").astype(str)

    def _load_rows(self, csv_path: str) -> List[dict]:
        df = pd.read_csv(csv_path)
        df = self._normalize_dataframe(df)
        return df.to_dict(orient="records")

    def load_rows_from_csv(self, csv_path: str) -> List[dict]:
        return self._load_rows(csv_path)

    def build_dataloaders_from_rows(self, train_rows: List[dict], val_rows: List[dict]) -> Tuple[DataLoader, DataLoader]:
        train_dataset = CausalLMDataset(
            rows=train_rows,
            tokenizer=self.tokenizer,
            max_length=self.config.max_length,
            reasoning_weight=self.config.reasoning_weight,
            code_weight=self.config.code_weight,
            spec_weight=self.config.spec_weight,
        )
        val_dataset = CausalLMDataset(
            rows=val_rows,
            tokenizer=self.tokenizer,
            max_length=self.config.max_length,
            reasoning_weight=self.config.reasoning_weight,
            code_weight=self.config.code_weight,
            spec_weight=self.config.spec_weight,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.train_batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=self.config.eval_batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
        )
        return train_loader, val_loader


def build_data_module(
    tokenizer: PreTrainedTokenizerBase,
    max_length: int,
    train_batch_size: int,
    eval_batch_size: int,
    num_workers: int,
    reasoning_weight: float = 1.0,
    code_weight: float = 1.0,
    spec_weight: float = 1.0,
) -> ReasoningDataModule:
    data_config = DataConfig(
        max_length=max_length,
        train_batch_size=train_batch_size,
        eval_batch_size=eval_batch_size,
        num_workers=num_workers,
        reasoning_weight=reasoning_weight,
        code_weight=code_weight,
        spec_weight=spec_weight,
    )
    return ReasoningDataModule(data_config, tokenizer)


## 5. Scheduler Helper

In [ ]:
def build_cosine_scheduler_with_warmup(
    optimizer: Optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
):
    num_training_steps = max(1, int(num_training_steps))
    num_warmup_steps = max(0, min(int(num_warmup_steps), num_training_steps))

    try:
        return get_cosine_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps,
        )
    except Exception:
        def lr_lambda(current_step: int) -> float:
            if current_step < num_warmup_steps:
                return float(current_step) / float(max(1, num_warmup_steps))
            if num_training_steps == num_warmup_steps:
                return 0.0
            progress = float(current_step - num_warmup_steps) / float(num_training_steps - num_warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        return LambdaLR(optimizer, lr_lambda=lr_lambda)


## 6. Safe Verifiers (Binary + Graded)

In [ ]:
EXECUTION_TIMEOUT_SECONDS = 2.0
CPU_LIMIT_SECONDS = 2
MEMORY_LIMIT_BYTES = 256 * 1024 * 1024
FILE_SIZE_LIMIT_BYTES = 1 * 1024 * 1024
MAX_STDOUT_CHARS = 50_000
MAX_STDERR_CHARS = 20_000
MAX_RESULT_REPR_CHARS = 20_000
MAX_PIPE_CHARS = 300_000

FORBIDDEN_IMPORT_ROOTS = {
    "os", "sys", "subprocess", "socket", "shutil", "pathlib", "multiprocessing", "signal"
}
FORBIDDEN_CALL_NAMES = {"exec", "eval", "compile", "__import__"}

ERROR_SYNTAX = "syntax_error"
ERROR_RUNTIME = "runtime_error"
ERROR_TIMEOUT = "timeout"
ERROR_MEMORY = "memory_error"
ERROR_UNSAFE = "unsafe_code"
ERROR_OK = "ok"

FLOAT_ATOL = 1e-8
FLOAT_RTOL = 1e-8


@dataclass
class ExecutionResult:
    status: str
    result: Any = None
    stdout: str = ""
    stderr: str = ""
    detail: str = ""


_RUNNER_CODE = r'''
import ast
import contextlib
import io
import json
import traceback
import sys


class LimitedBuffer(io.StringIO):
    def __init__(self, max_chars: int) -> None:
        super().__init__()
        self.max_chars = max_chars
        self.overflow = False

    def write(self, s):
        if not isinstance(s, str):
            s = str(s)
        current = self.tell()
        remaining = self.max_chars - current
        if remaining <= 0:
            self.overflow = True
            return len(s)
        if len(s) > remaining:
            super().write(s[:remaining])
            self.overflow = True
            return len(s)
        return super().write(s)


def _safe_repr(value, max_chars: int):
    text = repr(value)
    if len(text) <= max_chars:
        return text, False
    return text[:max_chars], True


def _emit(payload):
    sys.stdout.write(json.dumps(payload, ensure_ascii=False))
    sys.stdout.write("\n")
    sys.stdout.flush()


def _parse_call_input(input_text: str):
    raw = (input_text or "").strip()
    if not raw:
        return tuple(), {}

    try:
        parsed = ast.parse(f"_f({raw})", mode="eval")
        call_node = parsed.body
        if isinstance(call_node, ast.Call):
            args = [ast.literal_eval(arg) for arg in call_node.args]
            kwargs = {}
            for kw in call_node.keywords:
                if kw.arg is None:
                    raise ValueError("Unsupported **kwargs input.")
                kwargs[kw.arg] = ast.literal_eval(kw.value)
            return tuple(args), kwargs
    except Exception:
        pass

    try:
        val = ast.literal_eval(raw)
        if isinstance(val, tuple):
            return val, {}
        return (val,), {}
    except Exception:
        return (raw,), {}


def _run_function_mode(compiled, payload):
    function_name = payload.get("function_name") or ""
    input_text = payload.get("input_text") or ""
    max_output_chars = int(payload.get("max_output_chars", 50000))
    max_result_repr_chars = int(payload.get("max_result_repr_chars", 20000))

    namespace = {"__name__": "__candidate__"}
    stdout_buffer = LimitedBuffer(max_output_chars)
    stderr_buffer = LimitedBuffer(max_output_chars)
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exec(compiled, namespace, namespace)
        fn = namespace.get(function_name)
        if fn is None or not callable(fn):
            raise ValueError(f"Function '{function_name}' not found or not callable.")
        args, kwargs = _parse_call_input(input_text)
        result = fn(*args, **kwargs)

    result_repr, result_truncated = _safe_repr(result, max_result_repr_chars)
    return {
        "status": "ok",
        "result_repr": result_repr,
        "result_truncated": result_truncated,
        "stdout": stdout_buffer.getvalue(),
        "stderr": stderr_buffer.getvalue(),
        "stdout_overflow": stdout_buffer.overflow,
        "stderr_overflow": stderr_buffer.overflow,
    }


def _run_script_mode(compiled, payload):
    stdin_text = payload.get("stdin_text") or ""
    max_output_chars = int(payload.get("max_output_chars", 50000))
    namespace = {"__name__": "__main__"}
    stdout_buffer = LimitedBuffer(max_output_chars)
    stderr_buffer = LimitedBuffer(max_output_chars)
    fake_stdin = io.StringIO(stdin_text)

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        old_stdin = sys.stdin
        try:
            sys.stdin = fake_stdin
            exec(compiled, namespace, namespace)
        finally:
            sys.stdin = old_stdin

    return {
        "status": "ok",
        "stdout": stdout_buffer.getvalue(),
        "stderr": stderr_buffer.getvalue(),
        "stdout_overflow": stdout_buffer.overflow,
        "stderr_overflow": stderr_buffer.overflow,
    }


def main():
    payload_path = sys.argv[1]
    with open(payload_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    candidate_path = payload["candidate_path"]
    mode = payload.get("mode") or "function"
    with open(candidate_path, "r", encoding="utf-8") as f:
        candidate_code = f.read()

    try:
        compiled = compile(candidate_code, candidate_path, "exec")
    except SyntaxError as e:
        _emit({"status": "syntax_error", "error": str(e)})
        return
    except Exception as e:
        _emit({"status": "runtime_error", "error": str(e)})
        return

    try:
        if mode == "function":
            _emit(_run_function_mode(compiled, payload))
            return
        if mode == "script":
            _emit(_run_script_mode(compiled, payload))
            return
        _emit({"status": "runtime_error", "error": f"Unknown mode '{mode}'"})
    except MemoryError:
        _emit({"status": "memory_error", "error": "MemoryError"})
    except SyntaxError as e:
        _emit({"status": "syntax_error", "error": str(e)})
    except Exception as e:
        _emit({"status": "runtime_error", "error": str(e), "traceback": traceback.format_exc(limit=6)})


if __name__ == "__main__":
    main()
'''


def normalize_text(text: str) -> str:
    return " ".join(text.replace("```python", "```").replace("```", "").split()).strip().lower()


def extract_code_block(text: str) -> str:
    match = re.search(r"```(?:python)?\s*(.*?)```", text or "", flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return (text or "").strip()


def extract_first_function_name(code: str) -> Optional[str]:
    try:
        tree = ast.parse(code)
    except Exception:
        return None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            return node.name
    return None


def parse_examples(example_text: str) -> List[Tuple[str, str]]:
    text = (example_text or "").replace("\r\n", "\n").strip()
    if not text:
        return []

    pairs: List[Tuple[str, str]] = []
    try:
        pattern = r"Input:\s*(.*?)\s*(?:;)?\s*Output:\s*(.*?)(?=\n\s*Input:|$)"
        for inp, out in re.findall(pattern, text, flags=re.DOTALL | re.IGNORECASE):
            inp_clean = inp.strip()
            out_clean = out.strip()
            if inp_clean and out_clean:
                pairs.append((inp_clean, out_clean))
        if pairs:
            return pairs
    except Exception:
        pass

    return []


def parse_expected_output(output_text: str):
    try:
        return ast.literal_eval((output_text or "").strip())
    except Exception:
        return (output_text or "").strip()


def _is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def _numbers_close(a: float, b: float) -> bool:
    diff = abs(a - b)
    limit = FLOAT_ATOL + FLOAT_RTOL * max(abs(a), abs(b))
    return diff <= limit


def results_equal(predicted, expected) -> bool:
    if _is_number(predicted) and _is_number(expected):
        return _numbers_close(float(predicted), float(expected))

    if isinstance(predicted, str) and isinstance(expected, str):
        return normalize_text(predicted) == normalize_text(expected)

    if isinstance(predicted, (list, tuple)) and isinstance(expected, (list, tuple)):
        if len(predicted) != len(expected):
            return False
        return all(results_equal(p, e) for p, e in zip(predicted, expected))

    if isinstance(predicted, dict) and isinstance(expected, dict):
        if predicted.keys() != expected.keys():
            return False
        return all(results_equal(predicted[k], expected[k]) for k in predicted)

    return normalize_text(str(predicted)) == normalize_text(str(expected))


def _root_module_name(module_name: str) -> str:
    return (module_name or "").split(".", 1)[0]


def _called_name(func_node: ast.AST) -> Optional[str]:
    if isinstance(func_node, ast.Name):
        return func_node.id
    if isinstance(func_node, ast.Attribute):
        return func_node.attr
    return None


def _analyze_code_safety(code: str) -> Tuple[bool, str]:
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return False, ERROR_SYNTAX
    except Exception:
        return False, ERROR_RUNTIME

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if _root_module_name(alias.name) in FORBIDDEN_IMPORT_ROOTS:
                    return False, ERROR_UNSAFE
        elif isinstance(node, ast.ImportFrom):
            if _root_module_name(node.module or "") in FORBIDDEN_IMPORT_ROOTS:
                return False, ERROR_UNSAFE
        elif isinstance(node, ast.Call):
            called = _called_name(node.func)
            if called in FORBIDDEN_CALL_NAMES:
                return False, ERROR_UNSAFE

    return True, ERROR_OK


def _build_preexec_fn():
    if resource is None or os.name != "posix":
        return None

    def _set_limits():
        try:
            resource.setrlimit(resource.RLIMIT_CPU, (CPU_LIMIT_SECONDS, CPU_LIMIT_SECONDS))
        except Exception:
            pass
        try:
            resource.setrlimit(resource.RLIMIT_AS, (MEMORY_LIMIT_BYTES, MEMORY_LIMIT_BYTES))
        except Exception:
            pass
        try:
            resource.setrlimit(resource.RLIMIT_FSIZE, (FILE_SIZE_LIMIT_BYTES, FILE_SIZE_LIMIT_BYTES))
        except Exception:
            pass

    return _set_limits


def _build_safe_env() -> Dict[str, str]:
    env: Dict[str, str] = {
        "PYTHONIOENCODING": "utf-8",
        "PYTHONUNBUFFERED": "1",
    }
    for key in ("PATH", "SYSTEMROOT", "WINDIR", "HOME", "TMPDIR", "TEMP", "TMP", "LANG", "LC_ALL"):
        if key in os.environ:
            env[key] = os.environ[key]
    return env


def _truncate_text(text: str, max_chars: int) -> str:
    return text if len(text) <= max_chars else text[:max_chars]


def _parse_runner_payload(stdout_text: str) -> Optional[Dict[str, Any]]:
    for line in reversed(stdout_text.splitlines()):
        raw = line.strip()
        if not raw:
            continue
        try:
            payload = json.loads(raw)
            if isinstance(payload, dict):
                return payload
        except Exception:
            continue
    return None


def _deserialize_result_repr(result_repr: str):
    try:
        return ast.literal_eval(result_repr)
    except Exception:
        return result_repr


def _run_candidate_subprocess(code: str, mode: str, *, input_text: str = "", function_name: str = "") -> ExecutionResult:
    try:
        with tempfile.TemporaryDirectory() as tmp_dir:
            candidate_path = os.path.join(tmp_dir, "candidate.py")
            runner_path = os.path.join(tmp_dir, "_runner.py")
            payload_path = os.path.join(tmp_dir, "_payload.json")

            with open(candidate_path, "w", encoding="utf-8") as f:
                f.write(code)
            with open(runner_path, "w", encoding="utf-8") as f:
                f.write(_RUNNER_CODE)

            payload = {
                "candidate_path": candidate_path,
                "mode": mode,
                "function_name": function_name,
                "input_text": input_text,
                "stdin_text": input_text,
                "max_output_chars": MAX_STDOUT_CHARS,
                "max_result_repr_chars": MAX_RESULT_REPR_CHARS,
            }
            with open(payload_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False)

            run_kwargs: Dict[str, Any] = {
                "cwd": tmp_dir,
                "capture_output": True,
                "text": True,
                "timeout": EXECUTION_TIMEOUT_SECONDS,
                "env": _build_safe_env(),
                "close_fds": True,
            }
            preexec_fn = _build_preexec_fn()
            if preexec_fn is not None:
                run_kwargs["preexec_fn"] = preexec_fn

            try:
                completed = subprocess.run([sys.executable, runner_path, payload_path], **run_kwargs)
            except subprocess.TimeoutExpired:
                return ExecutionResult(status=ERROR_TIMEOUT, detail="execution_timeout")
    except Exception as e:
        return ExecutionResult(status=ERROR_RUNTIME, detail=f"subprocess_setup_error: {e}")

    stdout_text = _truncate_text(completed.stdout or "", MAX_PIPE_CHARS)
    stderr_text = _truncate_text(completed.stderr or "", MAX_PIPE_CHARS)
    payload = _parse_runner_payload(stdout_text)
    if payload is None:
        return ExecutionResult(status=ERROR_RUNTIME, stdout=stdout_text, stderr=stderr_text)

    status = str(payload.get("status", ERROR_RUNTIME))
    if status != ERROR_OK:
        return ExecutionResult(
            status=status,
            stdout=_truncate_text(str(payload.get("stdout", "")), MAX_STDOUT_CHARS),
            stderr=_truncate_text(str(payload.get("stderr", "")), MAX_STDERR_CHARS),
            detail=str(payload.get("error", "")),
        )

    if payload.get("stdout_overflow") or payload.get("stderr_overflow") or payload.get("result_truncated"):
        return ExecutionResult(status=ERROR_RUNTIME, detail="output_limit_exceeded")

    if mode == "function":
        result = _deserialize_result_repr(str(payload.get("result_repr", "")))
    else:
        result = _truncate_text(str(payload.get("stdout", "")), MAX_STDOUT_CHARS)

    return ExecutionResult(status=ERROR_OK, result=result)


def _strict_interface_requires_reference_name(example_text: str, reference_name: Optional[str]) -> bool:
    if not reference_name:
        return False
    pattern = rf"\b{re.escape(reference_name)}\s*\("
    return re.search(pattern, example_text or "", flags=re.IGNORECASE) is not None


def _choose_function_name(pred_name: Optional[str], ref_name: Optional[str], example_text: str) -> Optional[str]:
    if _strict_interface_requires_reference_name(example_text, ref_name):
        return ref_name
    if pred_name:
        return pred_name
    return ref_name


def _run_function_tests(pred_code: str, tests: List[Tuple[str, str]], function_name: Optional[str]) -> Tuple[bool, int, int]:
    if not function_name:
        return False, 0, len(tests)

    passed = 0
    for input_text, output_text in tests:
        expected = parse_expected_output(output_text)
        result = _run_candidate_subprocess(pred_code, "function", input_text=input_text, function_name=function_name)
        if result.status != ERROR_OK:
            return False, 0, len(tests)
        if results_equal(result.result, expected):
            passed += 1
    return True, passed, len(tests)


def _run_script_tests(pred_code: str, tests: List[Tuple[str, str]]) -> Tuple[bool, int, int]:
    passed = 0
    for input_text, output_text in tests:
        expected = parse_expected_output(output_text)
        result = _run_candidate_subprocess(pred_code, "script", input_text=input_text)
        if result.status != ERROR_OK:
            return False, 0, len(tests)
        if results_equal(result.result, expected):
            passed += 1
    return True, passed, len(tests)


class BinaryCodeVerifier:
    def __call__(self, prediction: str, reference: str, example: str) -> bool:
        try:
            pred_code = extract_code_block(prediction)
            safe, status = _analyze_code_safety(pred_code)
            if not safe or status == ERROR_SYNTAX:
                return False

            tests = parse_examples(example)
            if not tests:
                return False

            ref_code = extract_code_block(reference)
            pred_name = extract_first_function_name(pred_code)
            ref_name = extract_first_function_name(ref_code)

            fn_name = _choose_function_name(pred_name, ref_name, example)
            function_ok, function_passed, function_total = _run_function_tests(pred_code, tests, fn_name)
            if function_ok and function_passed == function_total:
                return True

            script_ok, script_passed, script_total = _run_script_tests(pred_code, tests)
            return script_ok and script_passed == script_total
        except Exception:
            return False


def _count_passed_tests(pred_code: str, reference: str, example: str) -> Tuple[int, int, bool]:
    tests = parse_examples(example)
    if not tests:
        return 0, 0, True

    ref_code = extract_code_block(reference)
    pred_name = extract_first_function_name(pred_code)
    ref_name = extract_first_function_name(ref_code)
    fn_name = _choose_function_name(pred_name, ref_name, example)

    function_ok, function_passed, function_total = _run_function_tests(pred_code, tests, fn_name)
    if function_ok:
        return function_passed, function_total, True

    script_ok, script_passed, script_total = _run_script_tests(pred_code, tests)
    if script_ok:
        return script_passed, script_total, True

    return 0, len(tests), False


class GradedCodeVerifier:
    def __call__(self, prediction: str, reference: str, example: str) -> float:
        try:
            pred_code = extract_code_block(prediction)
            safe, status = _analyze_code_safety(pred_code)
            if not safe or status == ERROR_SYNTAX:
                return 0.0

            passed, total, executed = _count_passed_tests(pred_code, reference, example)
            if total <= 0:
                return 0.4
            if not executed:
                return 0.0
            return 0.4 + 0.6 * (float(passed) / float(total))
        except Exception:
            return 0.0


## 7. Easy SFT Trainer

In [ ]:
@dataclass
class EasyTrainConfig:
    model_name: str = "gpt2"
    train_csv_path: str = "data/reasoning_dataset.csv"
    domain_csvs: str = "alg:data/alg_tasks.csv,math:data/math_tasks.csv,struct:data/struct_tasks.csv"
    output_dir: str = "artifacts_easy"
    epochs: int = 2
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0
    seed: int = 42
    max_length: int = 768
    train_batch_size: int = 2
    eval_batch_size: int = 2
    num_workers: int = 0
    reasoning_weight: float = 1.0
    code_weight: float = 1.0
    spec_weight: float = 1.0
    pass_k: int = 3
    pass_k_samples: int = 8
    eval_max_new_tokens: int = 256
    eval_temperature: float = 0.8
    eval_top_p: float = 0.95
    eval_max_rows: int = 0
    grad_accum_steps: int = 1


class EasyFineTuner:
    def __init__(self, config: EasyTrainConfig) -> None:
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.factory = ModelFactory(ModelConfig(model_name=config.model_name))
        self.model, self.tokenizer = self.factory.build()
        self.model.to(self.device)

        self.data_module = build_data_module(
            tokenizer=self.tokenizer,
            max_length=config.max_length,
            train_batch_size=config.train_batch_size,
            eval_batch_size=config.eval_batch_size,
            num_workers=config.num_workers,
            reasoning_weight=config.reasoning_weight,
            code_weight=config.code_weight,
            spec_weight=config.spec_weight,
        )

        self.binary_verifier = BinaryCodeVerifier()
        self.graded_verifier = GradedCodeVerifier()

        self.domain_map = parse_domain_csvs(config.domain_csvs)
        self.train_loader, self.eval_loader, self.eval_rows = self._build_loaders()

        accum_steps = max(1, config.grad_accum_steps)
        updates_per_epoch = math.ceil(len(self.train_loader) / accum_steps)
        total_steps = max(1, updates_per_epoch * config.epochs)
        warmup_steps = int(total_steps * config.warmup_ratio)

        self.optimizer = AdamW(self.model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        self.scheduler = build_cosine_scheduler_with_warmup(
            optimizer=self.optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        self.output_dir = Path(config.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.best_model_dir = self.output_dir / "best_model"
        self.best_model_dir.mkdir(parents=True, exist_ok=True)

    def _build_loaders(self):
        train_rows = self.data_module.load_rows_from_csv(self.config.train_csv_path)
        eval_rows: List[dict] = []
        for csv_path in self.domain_map.values():
            eval_rows.extend(self.data_module.load_rows_from_csv(csv_path))
        if not eval_rows:
            raise ValueError("Validation rows are empty. Check domain_csvs.")

        train_loader, eval_loader = self.data_module.build_dataloaders_from_rows(train_rows=train_rows, val_rows=eval_rows)
        return train_loader, eval_loader, eval_rows

    def _compute_weighted_loss(self, logits: torch.Tensor, labels: torch.Tensor, token_weights: torch.Tensor) -> torch.Tensor:
        aligned_logits = logits[:, :-1, :]
        aligned_labels = labels[:, 1:]
        aligned_weights = token_weights[:, 1:]

        vocab_size = aligned_logits.shape[-1]
        token_loss = F.cross_entropy(
            aligned_logits.reshape(-1, vocab_size),
            aligned_labels.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).reshape_as(aligned_labels)

        valid_mask = (aligned_labels != -100).float()
        aligned_weights = aligned_weights * valid_mask
        denom = aligned_weights.sum().clamp(min=1e-8)
        return (token_loss * aligned_weights).sum() / denom

    def _extract_response_ids(self, prompt_ids: torch.Tensor, generated_ids: torch.Tensor) -> torch.Tensor:
        return generated_ids[0][prompt_ids.shape[1]:]

    @staticmethod
    def _estimate_pass_at_k(n: int, c: int, k: int) -> float:
        if c <= 0:
            return 0.0
        if n - c < k:
            return 1.0
        product = 1.0
        for i in range(n - c + 1, n + 1):
            product *= 1.0 - (float(k) / float(i))
        return 1.0 - product

    @torch.no_grad()
    def evaluate_generation_metrics(self) -> Dict[str, float]:
        rows = self.eval_rows
        if self.config.eval_max_rows > 0:
            rows = rows[: self.config.eval_max_rows]
        if not rows:
            return {"pass@1": 0.0, f"pass@{self.config.pass_k}": 0.0, "avg_reward_score": 0.0}

        n_samples = max(1, self.config.pass_k_samples)
        k_value = max(1, min(self.config.pass_k, n_samples))

        pass1_scores: List[float] = []
        passk_scores: List[float] = []
        reward_scores: List[float] = []

        self.model.eval()
        progress = tqdm(rows, desc="eval generation", leave=False)
        for row in progress:
            prompt = PromptFormatter.format_generation_prompt(
                question=row["question"],
                example=row.get("example", ""),
            )
            encoded = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=self.config.max_length)
            input_ids = encoded["input_ids"].to(self.device)
            attention_mask = encoded["attention_mask"].to(self.device)

            passed = 0
            row_reward_sum = 0.0
            for _ in range(n_samples):
                generated = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=True,
                    top_p=self.config.eval_top_p,
                    temperature=self.config.eval_temperature,
                    max_new_tokens=self.config.eval_max_new_tokens,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                response_ids = self._extract_response_ids(input_ids, generated)
                prediction = self.tokenizer.decode(response_ids, skip_special_tokens=True)

                if self.binary_verifier(prediction, row["solution"], row.get("example", "")):
                    passed += 1
                row_reward_sum += self.graded_verifier(prediction, row["solution"], row.get("example", ""))

            pass1_scores.append(self._estimate_pass_at_k(n_samples, passed, 1))
            passk_scores.append(self._estimate_pass_at_k(n_samples, passed, k_value))
            reward_scores.append(row_reward_sum / float(n_samples))

        return {
            "pass@1": float(sum(pass1_scores) / len(pass1_scores)),
            f"pass@{k_value}": float(sum(passk_scores) / len(passk_scores)),
            "avg_reward_score": float(sum(reward_scores) / len(reward_scores)),
        }

    def _run_epoch(self, train: bool) -> float:
        loader = self.train_loader if train else self.eval_loader
        self.model.train() if train else self.model.eval()

        loss_sum = torch.tensor(0.0, device=self.device)
        step_count = torch.tensor(0.0, device=self.device)
        accum_steps = max(1, self.config.grad_accum_steps)

        progress = tqdm(loader, desc="train step" if train else "eval step", leave=False)
        if train:
            self.optimizer.zero_grad()

        for micro_step, batch in enumerate(progress, start=1):
            batch = {k: v.to(self.device) for k, v in batch.items()}
            token_weights = batch.pop("token_weights", None)

            with torch.set_grad_enabled(train):
                outputs = self.model(**batch)
                if token_weights is None:
                    loss = outputs.loss
                else:
                    loss = self._compute_weighted_loss(
                        logits=outputs.logits,
                        labels=batch["labels"],
                        token_weights=token_weights,
                    )

            if train:
                (loss / accum_steps).backward()
                is_update_step = (micro_step % accum_steps == 0) or (micro_step == len(loader))
                if is_update_step:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                    self.optimizer.step()
                    self.scheduler.step()
                    self.optimizer.zero_grad()

            loss_sum = loss_sum + loss.detach()
            step_count = step_count + 1.0

        return float(loss_sum.item() / max(1.0, step_count.item()))

    def train(self) -> Dict:
        best_eval_loss = float("inf")
        history: List[Dict] = []

        for epoch in range(1, self.config.epochs + 1):
            train_loss = self._run_epoch(train=True)
            eval_loss = self._run_epoch(train=False)
            eval_perplexity = float(math.exp(eval_loss)) if eval_loss < 50 else float("inf")

            print(
                f"Epoch {epoch}/{self.config.epochs} | "
                f"train_loss={train_loss:.4f} | eval_loss={eval_loss:.4f} | "
                f"eval_ppl={eval_perplexity:.2f}"
            )

            history.append(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "eval_loss": eval_loss,
                    "eval_perplexity": eval_perplexity,
                }
            )

            if eval_loss < best_eval_loss:
                best_eval_loss = eval_loss
                self.model.save_pretrained(self.best_model_dir)
                self.tokenizer.save_pretrained(self.best_model_dir)

        generation_metrics = self.evaluate_generation_metrics()

        result = {
            "model_name": self.config.model_name,
            "train_csv_path": self.config.train_csv_path,
            "domain_csvs": self.domain_map,
            "best_eval_loss": best_eval_loss,
            "generation_metrics": generation_metrics,
            "history": history,
            "best_model_dir": str(self.best_model_dir),
        }

        metrics_path = self.output_dir / "easy_train_metrics.json"
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"Best model saved to: {self.best_model_dir}")
        print(f"Metrics saved to: {metrics_path}")
        return result


def run_easy_finetuning(config: EasyTrainConfig) -> Dict:
    set_seed(config.seed)
    trainer = EasyFineTuner(config)
    return trainer.train()


## 8. Run Config and Start Training

In [ ]:
config = EasyTrainConfig(
    model_name="gpt2",
    train_csv_path="data/reasoning_dataset.csv",
    domain_csvs="alg:data/alg_tasks.csv,math:data/math_tasks.csv,struct:data/struct_tasks.csv",
    output_dir="artifacts_easy_gpt2",
    epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    seed=42,
    max_length=768,
    train_batch_size=2,
    eval_batch_size=2,
    num_workers=0,
    reasoning_weight=1.0,
    code_weight=1.0,
    spec_weight=1.0,
    pass_k=3,
    pass_k_samples=8,
    eval_max_new_tokens=256,
    eval_temperature=0.8,
    eval_top_p=0.95,
    eval_max_rows=0,
    grad_accum_steps=1,
)

result = run_easy_finetuning(config)
result
